# **Hands-on Practice: ABSA (Aspect-Based Sentiment Analysis)**

## **1. Load Dataset**

### (1) import library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import torch
from google.colab import files

from transformers import Trainer, TrainingArguments, AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset, Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import *
from sklearn.preprocessing import LabelEncoder

### (2) setting device

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### (3) load data

* Laptop_Review_with_aspect.csv

In [ ]:
uploaded = files.upload()
data_path = list(uploaded.keys())[0]

In [ ]:
data = pd.read_csv(data_path)
data.head()

In [ ]:
sns.countplot(x='aspect', data = data, hue = 'aspect')
plt.grid()
plt.xticks(rotation=90)
plt.show()

In [ ]:
sns.countplot(x='aspect', data = data, hue = 'label')
plt.grid()
plt.xticks(rotation=90)
plt.show()

## **2. Pre-processing Data**

### (1) Integer encoding : y

In [ ]:
le = LabelEncoder()
data['label'] = le.fit_transform(data['label'])
data.head()

In [ ]:
label_list = le.classes_
label_list

### (2) Pre-processing Input Data
* [ASPECT]aspect[SEP]review

In [ ]:
# data split



# Combine aspect and text into a single input format



# Keep only the text and label columns



### (3) Dataset

## **3. Tokenization**

### (1) download tokenizer

### (2) add special token

In [ ]:
# Define custom special tokens to mark the aspect in the input text


# Add the special tokens to the tokenizer's vocabulary



### (3) tokenization

In [ ]:
def preprocess_function(data):
    # Tokenize the input text with fixed max length

    # Attach the label to the tokenized output

    return tokenized

# Apply preprocessing to each example in the train and validation sets


## **4. Load Pre-trained Model**

In [ ]:
# Load the pre-trained model with 2 sentiment labels


# Resize embeddings to include the added special token


## **5. Set Training Arguments**

## **6. Fine-Tune Model**

## **7. Evaluate**

In [ ]:
want_keys = ["input_ids", "attention_mask"]              # keys the model expects (add "token_type_ids" for BERT if present)
batch = {k: torch.tensor(val_ts[k]) for k in want_keys}  # convert dataset columns (lists) into torch tensors
batch = {k: v.to(device) for k, v in batch.items()}      # move tensors to the target device (CPU/GPU)

with torch.no_grad():                                    # disable gradient tracking for inference
    outputs = model(**batch)                             # forward pass; batch must be a dict of tensors
logits = outputs.logits                                  # model scores, shape: [N, num_labels]
pred  = logits.argmax(dim=-1)                            # predicted class ids (argmax over label dimension)

In [ ]:
pred = pred.cpu().numpy()
print(confusion_matrix(val_ts['label'], pred))
print(classification_report(val_ts['label'], pred, target_names = label_list))

In [ ]:
val_org['pred'] = pred
val_org.head()

In [ ]:
# Get unique aspect categories in the validation set
asp = val_org['aspect'].unique()
asp

In [ ]:
results = []  # List to store metrics for each aspect

for aspect in asp:
    sub = val_org[val_org['aspect'] == aspect]  # Filter rows for the current aspect
    precision = precision_score(sub['label'], sub['pred'], zero_division=0)  # Aspect-wise precision
    recall = recall_score(sub['label'], sub['pred'], zero_division=0)        # Aspect-wise recall
    f1 = f1_score(sub['label'], sub['pred'], zero_division=0)                # Aspect-wise F1-score
    results.append({'aspect': aspect, 'precision': precision, 'recall': recall, 'f1': f1})

# Create a DataFrame of metrics and sort by aspect name
df_scores = pd.DataFrame(results).sort_values('aspect')

# Convert to long format for easier plotting with seaborn
df_melted = df_scores.melt(id_vars='aspect', var_name='metric', value_name='score')

In [ ]:
plt.figure(figsize=(10, 5))
# Draw a bar plot of scores for each aspect and metric
sns.barplot(data=df_melted, x='aspect', y='score', hue='metric')

plt.xticks(rotation=45)  # Rotate x-axis labels for readability
plt.ylim(0, 1.3)        # Set y-axis range to show bars clearly
plt.legend()
plt.grid()
plt.show()
